# Group DRO sur Waterbirds

Reproduction et analyse du papier **"Distributionally Robust Neural Networks for Group Shifts"** (Sagawa et al., 2020).

**Objectif** : montrer que l'ERM standard échoue sur les groupes minoritaires (ex: waterbird sur fond forêt) et que Group DRO + régularisation améliore la **worst-group accuracy**.

**Dataset Waterbirds** : oiseaux aquatiques/terrestres sur fonds eau/forêt. 4 groupes :
- Groupe 0 : landbird sur fond terre (majorité)
- Groupe 1 : landbird sur fond eau (minorité)
- Groupe 2 : waterbird sur fond terre (minorité)
- Groupe 3 : waterbird sur fond eau (majorité)

Le modèle ERM apprend le raccourci fond=eau → waterbird, et échoue sur les groupes 1 et 2.

In [ ]:
# Cell 1 — Setup & device
import os
import time
import csv
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.data.sampler import WeightedRandomSampler
import torchvision
import torchvision.transforms as transforms
from tqdm.notebook import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}, device: {device}")

def set_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(0)

In [ ]:
# Cell 2 — Télécharger le dataset Waterbirds
DATA_DIR = "/content/waterbirds"

if not os.path.exists(f"{DATA_DIR}/waterbird_complete95_forest2water2"):
    print("Téléchargement du dataset Waterbirds...")
    !wget -q https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz
    os.makedirs(DATA_DIR, exist_ok=True)
    !tar -xzf waterbird_complete95_forest2water2.tar.gz -C {DATA_DIR}
    !rm waterbird_complete95_forest2water2.tar.gz
    print("Dataset téléchargé.")
else:
    print("Dataset déjà présent.")

WB_DIR = f"{DATA_DIR}/waterbird_complete95_forest2water2"
!ls {WB_DIR} | head -10
!wc -l {WB_DIR}/metadata.csv

In [ ]:
# Cell 3 — Explorer le dataset et ses groupes
metadata = pd.read_csv(f"{WB_DIR}/metadata.csv")
print(f"Colonnes: {list(metadata.columns)}")
print(f"\nTaille totale: {len(metadata)}")
print(f"\nSplits (0=train, 1=val, 2=test):")
print(metadata['split'].value_counts().sort_index())

# y=0: landbird, y=1: waterbird
# place=0: land, place=1: water
GROUP_NAMES = {
    0: 'landbird / land bg',
    1: 'landbird / water bg',
    2: 'waterbird / land bg',
    3: 'waterbird / water bg',
}

# Compter les groupes dans le train set
train_meta = metadata[metadata['split'] == 0]
print(f"\nDistribution des groupes (train):")
for y in [0, 1]:
    for place in [0, 1]:
        g = y * 2 + place
        n = len(train_meta[(train_meta['y'] == y) & (train_meta['place'] == place)])
        print(f"  Groupe {g} ({GROUP_NAMES[g]}): {n:5d} ({n/len(train_meta)*100:.1f}%)")

In [ ]:
# Cell 4 — Visualiser des exemples de chaque groupe
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Exemples par groupe (2 par groupe)', fontsize=14)

for g in range(4):
    group_meta = train_meta[(train_meta['y'] == g // 2) & (train_meta['place'] == g % 2)]
    samples = group_meta.sample(2, random_state=42)
    for i, (_, row) in enumerate(samples.iterrows()):
        img = Image.open(os.path.join(WB_DIR, row['img_filename'])).convert('RGB')
        axes[i, g].imshow(img)
        axes[i, g].set_title(f"G{g}: {GROUP_NAMES[g]}\ny={row['y']}, place={row['place']}", fontsize=9)
        axes[i, g].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5 — Dataset Waterbirds compatible Group DRO

class WaterbirdsDataset(Dataset):
    """Dataset Waterbirds avec labels (x, y, group)."""

    def __init__(self, root_dir, split, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.metadata = pd.read_csv(os.path.join(root_dir, 'metadata.csv'))

        # split: 0=train, 1=val, 2=test
        split_map = {'train': 0, 'val': 1, 'test': 2}
        self.metadata = self.metadata[self.metadata['split'] == split_map[split]].reset_index(drop=True)

        self.y_array = self.metadata['y'].values
        self.place_array = self.metadata['place'].values
        self.group_array = self.y_array * 2 + self.place_array  # 4 groupes
        self.filename_array = self.metadata['img_filename'].values

        self.n_classes = 2
        self.n_groups = 4
        self._group_counts = torch.zeros(self.n_groups)
        for g in range(self.n_groups):
            self._group_counts[g] = (self.group_array == g).sum()

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.filename_array[idx])
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        y = self.y_array[idx]
        g = self.group_array[idx]
        return img, y, g

    def group_counts(self):
        return self._group_counts

    def group_str(self, g):
        return GROUP_NAMES[g]

    def get_loader(self, batch_size, train=False, reweight_groups=False):
        if not train:
            return DataLoader(self, batch_size=batch_size, shuffle=False,
                              num_workers=4, pin_memory=True)
        if reweight_groups:
            group_weights = len(self) / self._group_counts
            weights = group_weights[torch.LongTensor(self.group_array)]
            sampler = WeightedRandomSampler(weights, len(self), replacement=True)
            return DataLoader(self, batch_size=batch_size, sampler=sampler,
                              num_workers=4, pin_memory=True)
        return DataLoader(self, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True)

print("WaterbirdsDataset défini.")

In [ ]:
# Cell 6 — Transforms & DataLoaders
scale = 256.0 / 224.0
target_resolution = (224, 224)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(target_resolution, scale=(0.7, 1.0),
                                  ratio=(0.75, 4./3.), interpolation=2),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((int(224 * scale), int(224 * scale))),
    transforms.CenterCrop(target_resolution),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = WaterbirdsDataset(WB_DIR, 'train', transform=train_transform)
val_ds   = WaterbirdsDataset(WB_DIR, 'val',   transform=eval_transform)
test_ds  = WaterbirdsDataset(WB_DIR, 'test',  transform=eval_transform)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"\nTrain group counts:")
for g in range(4):
    print(f"  {GROUP_NAMES[g]}: {int(train_ds.group_counts()[g])}")

In [ ]:
# Cell 7 — LossComputer (coeur de Group DRO)

class LossComputer:
    """Calcule la loss ERM ou Group DRO.

    - ERM: loss = moyenne sur tous les exemples
    - Group DRO (robust=True): loss = somme pondérée des loss par groupe,
      où les poids adversariaux sont mis à jour par exponentiated gradient.
    """

    def __init__(self, criterion, n_groups, group_counts, is_robust=False,
                 alpha=0.2, gamma=0.1, adj=None, step_size=0.01):
        self.criterion = criterion
        self.is_robust = is_robust
        self.n_groups = n_groups
        self.group_counts = group_counts.to(device)
        self.group_frac = self.group_counts / self.group_counts.sum()
        self.alpha = alpha
        self.gamma = gamma
        self.step_size = step_size

        self.adj = torch.zeros(n_groups).float().to(device) if adj is None else adj.to(device)

        # Poids adversariaux (uniformes au départ)
        self.adv_probs = torch.ones(n_groups).to(device) / n_groups
        self.exp_avg_loss = torch.zeros(n_groups).to(device)
        self.exp_avg_initialized = torch.zeros(n_groups).byte().to(device)

        self.reset_stats()

    def loss(self, yhat, y, group_idx, is_training=False):
        per_sample_losses = self.criterion(yhat, y)
        group_loss, group_count = self._compute_group_avg(per_sample_losses, group_idx)
        group_acc, _ = self._compute_group_avg((yhat.argmax(1) == y).float(), group_idx)

        self._update_exp_avg_loss(group_loss, group_count)

        if self.is_robust:
            actual_loss, weights = self._compute_robust_loss(group_loss)
        else:
            actual_loss = per_sample_losses.mean()
            weights = None

        self._update_stats(actual_loss, group_loss, group_acc, group_count, weights)
        return actual_loss

    def _compute_robust_loss(self, group_loss):
        adjusted_loss = group_loss
        if torch.all(self.adj > 0):
            adjusted_loss = adjusted_loss + self.adj / torch.sqrt(self.group_counts)
        self.adv_probs = self.adv_probs * torch.exp(self.step_size * adjusted_loss.data)
        self.adv_probs = self.adv_probs / self.adv_probs.sum()
        robust_loss = group_loss @ self.adv_probs
        return robust_loss, self.adv_probs

    def _compute_group_avg(self, losses, group_idx):
        group_map = (group_idx == torch.arange(self.n_groups).unsqueeze(1).long().to(device)).float()
        group_count = group_map.sum(1)
        group_denom = group_count + (group_count == 0).float()
        group_loss = (group_map @ losses.view(-1)) / group_denom
        return group_loss, group_count

    def _update_exp_avg_loss(self, group_loss, group_count):
        prev_weights = (1 - self.gamma * (group_count > 0).float()) * (self.exp_avg_initialized > 0).float()
        curr_weights = 1 - prev_weights
        self.exp_avg_loss = self.exp_avg_loss * prev_weights + group_loss * curr_weights
        self.exp_avg_initialized = (self.exp_avg_initialized > 0) + (group_count > 0)

    def reset_stats(self):
        self.processed_data_counts = torch.zeros(self.n_groups).to(device)
        self.avg_group_loss = torch.zeros(self.n_groups).to(device)
        self.avg_group_acc = torch.zeros(self.n_groups).to(device)
        self.avg_actual_loss = 0.
        self.avg_acc = 0.
        self.batch_count = 0.

    def _update_stats(self, actual_loss, group_loss, group_acc, group_count, weights=None):
        denom = self.processed_data_counts + group_count
        denom += (denom == 0).float()
        prev_weight = self.processed_data_counts / denom
        curr_weight = group_count / denom
        self.avg_group_loss = prev_weight * self.avg_group_loss + curr_weight * group_loss
        self.avg_group_acc = prev_weight * self.avg_group_acc + curr_weight * group_acc
        denom_batch = self.batch_count + 1
        self.avg_actual_loss = (self.batch_count / denom_batch) * self.avg_actual_loss + (1 / denom_batch) * actual_loss.item()
        self.processed_data_counts += group_count
        self.batch_count += 1
        group_frac = self.processed_data_counts / (self.processed_data_counts.sum())
        self.avg_acc = (group_frac @ self.avg_group_acc).item()

    def get_summary(self):
        return {
            'avg_acc': self.avg_acc,
            'worst_acc': self.avg_group_acc.min().item(),
            'avg_loss': self.avg_actual_loss,
            'group_acc': self.avg_group_acc.cpu().numpy().copy(),
            'group_loss': self.avg_group_loss.cpu().numpy().copy(),
        }

print("LossComputer défini.")

In [ ]:
# Cell 8 — Fonctions d'entraînement et d'évaluation

def run_epoch(model, optimizer, loader, loss_computer, is_training, show_progress=True):
    if is_training:
        model.train()
    else:
        model.eval()

    iterator = tqdm(loader, leave=False) if show_progress else loader

    with torch.set_grad_enabled(is_training):
        for x, y, g in iterator:
            x, y, g = x.to(device), y.to(device), g.to(device)
            outputs = model(x)
            loss = loss_computer.loss(outputs, y, g, is_training)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

    return loss_computer.get_summary()


def train_model(model, train_loader, val_loader, test_loader, train_ds,
                n_epochs, lr, weight_decay, is_robust=False, gamma=0.1,
                reweight_groups=False, alpha=0.2, step_size=0.01,
                adj=None, log_every=10, save_best=True):
    """Boucle d'entraînement complète. Retourne l'historique."""

    optimizer = torch.optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, momentum=0.9, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(reduction='none')

    history = {'train': [], 'val': [], 'test': []}
    best_worst_acc = -1
    best_model_state = None

    for epoch in range(1, n_epochs + 1):
        t0 = time.time()

        # Train
        train_lc = LossComputer(criterion, 4, train_ds.group_counts(),
                                is_robust=is_robust, gamma=gamma,
                                alpha=alpha, step_size=step_size, adj=adj)
        train_summary = run_epoch(model, optimizer, train_loader, train_lc, is_training=True)
        history['train'].append(train_summary)

        # Val
        val_lc = LossComputer(criterion, 4, val_ds.group_counts(),
                              is_robust=is_robust, alpha=alpha, step_size=step_size)
        val_summary = run_epoch(model, optimizer, val_loader, val_lc, is_training=False)
        history['val'].append(val_summary)

        # Test
        test_lc = LossComputer(criterion, 4, test_ds.group_counts(),
                               is_robust=is_robust, alpha=alpha, step_size=step_size)
        test_summary = run_epoch(model, optimizer, test_loader, test_lc, is_training=False)
        history['test'].append(test_summary)

        elapsed = time.time() - t0

        # Early stopping sur worst-group val accuracy
        if save_best and val_summary['worst_acc'] > best_worst_acc:
            best_worst_acc = val_summary['worst_acc']
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch

        if epoch % log_every == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{n_epochs} ({elapsed:.0f}s) | "
                  f"Train loss: {train_summary['avg_loss']:.3f} | "
                  f"Val avg: {val_summary['avg_acc']*100:.1f}% worst: {val_summary['worst_acc']*100:.1f}% | "
                  f"Test avg: {test_summary['avg_acc']*100:.1f}% worst: {test_summary['worst_acc']*100:.1f}%")

    if save_best and best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nBest model restauré (epoch {best_epoch}, val worst-group acc: {best_worst_acc*100:.1f}%)")

    return history

print("Fonctions d'entraînement définies.")

---
## Expérience 1 : ERM (baseline)

Entraînement standard avec la loss cross-entropy moyennée sur tous les exemples. Le modèle va apprendre la corrélation spurieuse fond ↔ label.

In [ ]:
# Cell 9 — ERM : modèle + entraînement
set_seed(0)

BATCH_SIZE = 128
N_EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-4

# ResNet-50 pretrained, head binaire
model_erm = torchvision.models.resnet50(pretrained=True)
model_erm.fc = nn.Linear(model_erm.fc.in_features, 2)
model_erm = model_erm.to(device)

train_loader_erm = train_ds.get_loader(BATCH_SIZE, train=True, reweight_groups=False)
val_loader   = val_ds.get_loader(BATCH_SIZE, train=False)
test_loader  = test_ds.get_loader(BATCH_SIZE, train=False)

print(f"ERM : {N_EPOCHS} epochs, lr={LR}, wd={WEIGHT_DECAY}, batch={BATCH_SIZE}")
history_erm = train_model(
    model_erm, train_loader_erm, val_loader, test_loader, train_ds,
    n_epochs=N_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
    is_robust=False, log_every=10)

In [ ]:
# Cell 10 — Résultats ERM détaillés
def print_results(history, label, split='test'):
    last = history[split][-1]
    print(f"\n=== {label} — {split} set ===")
    print(f"Average accuracy:     {last['avg_acc']*100:.1f}%")
    print(f"Worst-group accuracy: {last['worst_acc']*100:.1f}%")
    print(f"\nPar groupe:")
    for g in range(4):
        print(f"  {GROUP_NAMES[g]:25s}: {last['group_acc'][g]*100:.1f}%")

print_results(history_erm, 'ERM', 'val')
print_results(history_erm, 'ERM', 'test')

---
## Expérience 2 : Group DRO

Optimisation robuste distributionnelle : on minimise la **pire loss parmi les 4 groupes** au lieu de la loss moyenne. Le modèle est forcé d'être bon sur tous les groupes, y compris les minoritaires.

In [ ]:
# Cell 11 — Group DRO : modèle + entraînement
set_seed(0)

model_dro = torchvision.models.resnet50(pretrained=True)
model_dro.fc = nn.Linear(model_dro.fc.in_features, 2)
model_dro = model_dro.to(device)

# Group DRO utilise le reweighting pour équilibrer les groupes dans les batches
train_loader_dro = train_ds.get_loader(BATCH_SIZE, train=True, reweight_groups=True)

print(f"Group DRO : {N_EPOCHS} epochs, lr={LR}, wd={WEIGHT_DECAY}, gamma=0.1, alpha=0.2")
history_dro = train_model(
    model_dro, train_loader_dro, val_loader, test_loader, train_ds,
    n_epochs=N_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
    is_robust=True, gamma=0.1, alpha=0.2, step_size=0.01,
    reweight_groups=True, log_every=10)

In [ ]:
# Cell 12 — Résultats Group DRO détaillés
print_results(history_dro, 'Group DRO', 'val')
print_results(history_dro, 'Group DRO', 'test')

---
## Expérience 3 : Impact du Weight Decay

Le papier montre que la régularisation (weight decay) est **critique** pour Group DRO. Testons avec un weight decay plus fort.

In [ ]:
# Cell 13 — Group DRO avec weight decay fort
set_seed(0)

WEIGHT_DECAY_STRONG = 1e-2  # 100x plus fort

model_dro_wd = torchvision.models.resnet50(pretrained=True)
model_dro_wd.fc = nn.Linear(model_dro_wd.fc.in_features, 2)
model_dro_wd = model_dro_wd.to(device)

train_loader_dro2 = train_ds.get_loader(BATCH_SIZE, train=True, reweight_groups=True)

print(f"Group DRO + strong wd : {N_EPOCHS} epochs, lr={LR}, wd={WEIGHT_DECAY_STRONG}")
history_dro_wd = train_model(
    model_dro_wd, train_loader_dro2, val_loader, test_loader, train_ds,
    n_epochs=N_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY_STRONG,
    is_robust=True, gamma=0.1, alpha=0.2, step_size=0.01,
    reweight_groups=True, log_every=10)

In [ ]:
# Cell 14 — Résultats Group DRO + strong wd
print_results(history_dro_wd, 'Group DRO + strong wd', 'val')
print_results(history_dro_wd, 'Group DRO + strong wd', 'test')

---
## Comparaison et visualisation

In [ ]:
# Cell 15 — Tableau comparatif

experiments = [
    ('ERM', history_erm),
    ('Group DRO', history_dro),
    ('Group DRO + strong wd', history_dro_wd),
]

print(f"{'Méthode':30s} | {'Avg acc':>8s} | {'Worst acc':>10s} | ", end='')
for g in range(4):
    print(f"{'G'+str(g):>6s}", end=' | ')
print()
print('-' * 100)

for name, hist in experiments:
    last = hist['test'][-1]
    print(f"{name:30s} | {last['avg_acc']*100:>7.1f}% | {last['worst_acc']*100:>9.1f}% | ", end='')
    for g in range(4):
        print(f"{last['group_acc'][g]*100:>5.1f}%", end=' | ')
    print()

In [ ]:
# Cell 16 — Courbes d'entraînement : avg accuracy vs worst-group accuracy

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, hist) in zip(axes, experiments):
    epochs = range(1, len(hist['val']) + 1)
    val_avg = [h['avg_acc'] * 100 for h in hist['val']]
    val_worst = [h['worst_acc'] * 100 for h in hist['val']]
    test_avg = [h['avg_acc'] * 100 for h in hist['test']]
    test_worst = [h['worst_acc'] * 100 for h in hist['test']]

    ax.plot(epochs, val_avg, label='Val avg', color='steelblue')
    ax.plot(epochs, val_worst, label='Val worst-group', color='steelblue', linestyle='--')
    ax.plot(epochs, test_avg, label='Test avg', color='coral')
    ax.plot(epochs, test_worst, label='Test worst-group', color='coral', linestyle='--')
    ax.set_title(name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 105)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Average vs Worst-Group Accuracy', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 17 — Bar chart comparatif : accuracy par groupe

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(4)
width = 0.25
colors = ['steelblue', 'coral', 'mediumseagreen']

for i, (name, hist) in enumerate(experiments):
    accs = hist['test'][-1]['group_acc'] * 100
    bars = ax.bar(x + i * width, accs, width, label=name, color=colors[i])
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels([f'G{g}\n{GROUP_NAMES[g]}' for g in range(4)], fontsize=9)
ax.set_ylabel('Test Accuracy (%)')
ax.set_ylim(0, 115)
ax.set_title('Accuracy par groupe — ERM vs Group DRO')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 18 — Visualiser les erreurs du modèle ERM sur le worst-group

model_erm.eval()
errors = []  # (img_path, true_y, pred_y, group)

with torch.no_grad():
    for idx in range(len(test_ds)):
        x, y, g = test_ds[idx]
        pred = model_erm(x.unsqueeze(0).to(device)).argmax(1).item()
        if pred != y:
            errors.append((test_ds.filename_array[idx], y, pred, g))

# Compter les erreurs par groupe
print("Erreurs ERM par groupe:")
from collections import Counter
error_counts = Counter(e[3] for e in errors)
for g in range(4):
    total = int(test_ds.group_counts()[g])
    n_err = error_counts.get(g, 0)
    print(f"  {GROUP_NAMES[g]:25s}: {n_err}/{total} erreurs ({n_err/total*100:.1f}%)")

# Afficher quelques erreurs du worst-group (waterbird / land bg)
worst_errors = [e for e in errors if e[3] == 2]  # group 2 = waterbird on land
n_show = min(8, len(worst_errors))

if n_show > 0:
    fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
    if n_show == 1:
        axes = [axes]
    for i in range(n_show):
        img_path, true_y, pred_y, g = worst_errors[i]
        img = Image.open(os.path.join(WB_DIR, img_path)).convert('RGB')
        axes[i].imshow(img)
        bird_names = ['landbird', 'waterbird']
        axes[i].set_title(f'True: {bird_names[true_y]}\nPred: {bird_names[pred_y]}', fontsize=9, color='red')
        axes[i].axis('off')
    plt.suptitle('Erreurs ERM — Waterbird on land background (worst group)', fontsize=12)
    plt.tight_layout()
    plt.show()

---
## Expérience bonus : Sweep sur le weight decay

Le papier insiste sur l'importance de la régularisation. Testons plusieurs valeurs pour quantifier cet effet.

In [ ]:
# Cell 19 — Sweep weight decay pour Group DRO
WD_VALUES = [0, 1e-4, 1e-3, 1e-2, 1e-1]
sweep_results = {}

for wd in WD_VALUES:
    set_seed(0)
    model_sweep = torchvision.models.resnet50(pretrained=True)
    model_sweep.fc = nn.Linear(model_sweep.fc.in_features, 2)
    model_sweep = model_sweep.to(device)

    loader = train_ds.get_loader(BATCH_SIZE, train=True, reweight_groups=True)

    print(f"\n--- Weight decay = {wd} ---")
    hist = train_model(
        model_sweep, loader, val_loader, test_loader, train_ds,
        n_epochs=N_EPOCHS, lr=LR, weight_decay=wd,
        is_robust=True, gamma=0.1, alpha=0.2, step_size=0.01,
        reweight_groups=True, log_every=25)
    sweep_results[wd] = hist

    last = hist['test'][-1]
    print(f"  Test avg: {last['avg_acc']*100:.1f}% | worst: {last['worst_acc']*100:.1f}%")

In [ ]:
# Cell 20 — Graphique : worst-group accuracy en fonction du weight decay

fig, ax = plt.subplots(figsize=(8, 5))

wds = list(sweep_results.keys())
avg_accs = [sweep_results[wd]['test'][-1]['avg_acc'] * 100 for wd in wds]
worst_accs = [sweep_results[wd]['test'][-1]['worst_acc'] * 100 for wd in wds]

wd_labels = [str(wd) for wd in wds]
x = np.arange(len(wds))

ax.plot(x, avg_accs, 'o-', label='Average accuracy', color='steelblue', markersize=8)
ax.plot(x, worst_accs, 's-', label='Worst-group accuracy', color='coral', markersize=8)

for i in range(len(wds)):
    ax.annotate(f'{avg_accs[i]:.1f}%', (x[i], avg_accs[i]), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=8, color='steelblue')
    ax.annotate(f'{worst_accs[i]:.1f}%', (x[i], worst_accs[i]), textcoords='offset points',
                xytext=(0, -15), ha='center', fontsize=8, color='coral')

ax.set_xticks(x)
ax.set_xticklabels(wd_labels)
ax.set_xlabel('Weight Decay')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Impact du Weight Decay sur Group DRO (Waterbirds)')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

---
## Résumé

| Méthode | Avg acc | Worst-group acc | Observation |
|---------|---------|-----------------|-------------|
| ERM | ~97% | ~60-70% | Apprend la corrélation fond ↔ label |
| Group DRO (wd=1e-4) | ~93% | ~75-80% | Améliore worst-group mais insuffisant |
| Group DRO (wd=1e-2) | ~90% | ~85-90% | Régularisation forte = meilleur worst-group |

**Conclusion clé** : Group DRO seul ne suffit pas — c'est le couplage DRO + régularisation forte qui permet d'améliorer la worst-group accuracy de 10-40 points.